# RecursiveJsonSplitter

`RecursiveJsonSplitter`는 **JSON 데이터**를 구조를 보존하면서 분할합니다.

## 주요 특징

- **깊이 우선 탐색**: JSON 구조를 재귀적으로 탐색하며 분할
- **구조 보존**: 가능한 한 중첩된 JSON 객체를 그대로 유지
- **크기 제한**: `min_chunk_size`와 `max_chunk_size` 사이로 청크 크기 조정

## 동작 원리

1. **재귀적 탐색**: JSON 트리를 깊이 우선으로 탐색
2. **객체 유지**: 중첩된 JSON 객체를 가능한 한 분할하지 않음
3. **크기 관리**: 청크가 너무 크면 필요에 따라 분할

## 사용 시나리오

- ✅ **API 응답 처리**: 복잡한 JSON API 응답 분할
- ✅ **JSON 문서**: 대규모 JSON 설정 파일, 데이터
- ✅ **구조화된 데이터**: JSON 형식의 로그, 메타데이터 등


# 08. RecursiveJsonSplitter — Chunking 섹션 마무리

## 학습 목표
- `RecursiveJsonSplitter`로 JSON 구조를 보존하면서 크기 제한 내로 분할해요
- `split_json()`과 `create_documents()`의 차이를 이해해요
- 6-2 섹션 전체 Chunking 전략을 데이터 유형별로 선택하는 기준을 정리해요

## 사전 지식
- 09-JSON-Loader 완료 (JSONLoader로 JSON 로딩)
- JSON 중첩 구조 이해

---

> **왜 JSON 전용 분할기가 필요한가?**
>
> `RecursiveCharacterTextSplitter`로 JSON을 분할하면 `{`, `}` 괄호가 맞지 않아 유효하지 않은 JSON 조각이 생길 수 있어요. `RecursiveJsonSplitter`는 **JSON 트리 구조를 유지**하면서 분할해요.


## 1. 기본 JSON 분할


In [ ]:
# ============================================================
# TODO: RecursiveJsonSplitter로 JSON 데이터를 구조를 보존하면서 분할해보세요
# 힌트: RecursiveJsonSplitter(max_chunk_size=300) → splitter.split_json(json_data=json_data)
# 예상 결과: 5개의 JSON 청크가 생성되고 각 청크가 유효한 JSON 구조를 유지합니다
# ============================================================

from langchain_text_splitters import RecursiveJsonSplitter
import json

# 샘플 JSON 데이터 (API 응답 예시)
json_data = {
    "api": "LangChain API",
    "version": "1.0",
    "endpoints": {
        "chat": {
            "url": "/api/chat",
            "methods": ["POST"],
            "description": "채팅 완성 API",
            "parameters": {
                "model": {
                    "type": "string",
                    "required": True,
                    "options": ["gpt-4", "gpt-3.5-turbo"]
                },
                "messages": {
                    "type": "array",
                    "required": True,
                    "description": "대화 메시지 배열"
                },
                "temperature": {
                    "type": "number",
                    "default": 0.7,
                    "range": [0, 2]
                }
            }
        },
        "embedding": {
            "url": "/api/embeddings",
            "methods": ["POST"],
            "description": "텍스트 임베딩 생성",
            "parameters": {
                "input": {
                    "type": "string",
                    "required": True
                },
                "model": {
                    "type": "string",
                    "default": "text-embedding-ada-002"
                }
            }
        }
    },
    "authentication": {
        "type": "Bearer Token",
        "header": "Authorization",
        "description": "API 키를 Bearer 토큰으로 전달"
    }
}

# 1단계: RecursiveJsonSplitter 생성
# - max_chunk_size=300: 각 청크의 최대 크기 (JSON 직렬화 후 문자 수)
# - 일반 TextSplitter와 달리 JSON 트리 구조를 탐색하며 분할

# 2단계: JSON 데이터 분할
# - split_json(): JSON 딕셔너리를 입력받아 딕셔너리 리스트로 분할


## 2. Document 객체로 변환


In [ ]:
# JSON 데이터를 Document 객체로 변환


## 핵심 정리 및 마무리

### Chunking 전략 선택 의사결정 트리

```mermaid
flowchart TD
    A{데이터 유형은?}:::process --> B[일반 텍스트]:::input
    A --> C[코드]:::input
    A --> D[Markdown<br/>HTML]:::input
    A --> E[JSON]:::input

    B --> F{품질 vs 속도?}:::process
    F --> G[속도 우선<br/>RecursiveCharacterTextSplitter]:::output
    F --> H[품질 우선<br/>SemanticChunker]:::output

    C --> I[from_language<br/>CodeSplitter]:::output

    D --> J[MarkdownHeader /<br/>HTMLHeaderTextSplitter]:::output

    E --> K[RecursiveJsonSplitter]:::output

    G --> L{토큰 제한<br/>엄격?}:::process
    L --> M[TokenTextSplitter]:::output
    L --> N[RecursiveCharacterTextSplitter]:::output

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

### 실무 팁
> 대부분의 RAG 프로젝트는 **RecursiveCharacterTextSplitter**로 시작하는 게 좋아요. 검색 품질이 부족하다고 느낄 때 SemanticChunker로 업그레이드하고, 토큰 비용이 문제라면 TokenTextSplitter를 고려해 보세요.

---

## 6-2 Chunking 섹션 마무리

이번 섹션에서 배운 8가지 분할 방식을 정리해 볼게요.

| 분할기 | 핵심 특징 | 추천 상황 |
|--------|----------|----------|
| CharacterTextSplitter | 단일 구분자, 단순 분할 | 구분자가 명확한 문서 |
| RecursiveCharacterTextSplitter | 다중 구분자, 재귀적 분할 | **대부분의 경우 (기본값)** |
| TokenTextSplitter | 토큰 수 기반 정확한 분할 | LLM API 토큰 제한 준수 |
| SemanticChunker | 임베딩 유사도 기반 분할 | 검색 품질이 최우선일 때 |
| CodeSplitter | 언어 문법 인식 분할 | 프로그래밍 코드 |
| MarkdownHeaderTextSplitter | 헤더 구조 보존 | Markdown 기술 문서 |
| HTMLHeaderTextSplitter | HTML 태그 구조 보존 | 웹 페이지 크롤링 |
| RecursiveJsonSplitter | JSON 구조 보존 | API 응답, JSON 파일 |

---

## 다음 단계

청킹까지 완료했어요. 이제 **6-3 Embeddings** 섹션으로 넘어가요. 분할된 청크를 벡터로 변환해서 검색 가능하게 만드는 방법을 배울게요.
